# Week 5 – Support Vector Machines

## Objective

This notebook applies Support Vector Machines (SVMs) to the NBA game total prediction project.

The goal is to predict whether an NBA game will finish over or under the sportsbook's pregame total.

The analysis will examine:

- Linear Support Vector Machines
- The kernel trick
- Nonlinear SVM models
- Regularization using the SVM `C` parameter

The binary target is:

- `1` = Actual game total is greater than the sportsbook total (Over)
- `0` = Actual game total is less than the sportsbook total (Under)

Games where the actual total exactly equals the sportsbook total will be excluded as pushes.

In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.dummy import DummyClassifier

In [20]:
merged = pd.read_csv(
    "../data/processed/merged_games_betting.csv",
    low_memory=False
)

In [21]:
classification_df = merged[merged["actual_total"] != merged["total"]].copy()

classification_df["over"] = (classification_df["actual_total"] > classification_df["total"]).astype(int)

In [22]:
features = [
    "total",
    "spread",
    "moneyline_home",
    "moneyline_away",
    "season"
]

svm_df = classification_df[features + ["over"]].dropna()

X = svm_df[features]
y = svm_df["over"]

print("Shape:", svm_df.shape)
print("\nClass balance:")
print(y.value_counts(normalize=True))

Shape: (18420, 6)

Class balance:
over
0    0.501466
1    0.498534
Name: proportion, dtype: float64


In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (14736, 5)
X_test: (3684, 5)


## Linear SVM:

In [24]:
svm_linear = SVC(
    kernel="linear",
    C=1.0,
    probability=True,
    random_state=42
)

svm_linear.fit(X_train_scaled, y_train)

y_pred_linear = svm_linear.predict(X_test_scaled)
y_score_linear = svm_linear.decision_function(X_test_scaled)

In [26]:
print("Accuracy:", accuracy_score(y_test, y_pred_linear))
print("Precision:", precision_score(y_test, y_pred_linear))
print("Recall:", recall_score(y_test, y_pred_linear))
print("F1 Score:", f1_score(y_test, y_pred_linear))
print("ROC AUC:", roc_auc_score(y_test, y_score_linear))

Accuracy: 0.500814332247557
Precision: 0.4994824016563147
Recall: 0.5253130103429504
F1 Score: 0.5120721676837358
ROC AUC: 0.506670028550469


## Kernel Trick:

In [27]:
svm_rbf = SVC(
    kernel="rbf",
    C=1.0,
    probability=True,
    random_state=42
)

svm_rbf.fit(X_train_scaled, y_train)

y_pred_rbf = svm_rbf.predict(X_test_scaled)
y_score_rbf = svm_rbf.decision_function(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred_rbf))
print("Precision:", precision_score(y_test, y_pred_rbf))
print("Recall:", recall_score(y_test, y_pred_rbf))
print("F1 Score:", f1_score(y_test, y_pred_rbf))
print("ROC AUC:", roc_auc_score(y_test, y_score_rbf))

Accuracy: 0.510043431053203
Precision: 0.5144927536231884
Recall: 0.30919978225367445
F1 Score: 0.3862631757905474
ROC AUC: 0.5168688561745436


In [28]:
rbf_scores = svm_rbf.decision_function(X_test_scaled)

print(
    "ROC AUC using decision function:",
    roc_auc_score(y_test, rbf_scores)
)

ROC AUC using decision function: 0.5168688561745436


## SVM Regularization with C:

In [29]:
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf"))
])

svm_cv = GridSearchCV(
    svm_pipeline,
    param_grid={
        "svm__C": [0.01, 0.1, 1, 10, 100]
    },
    cv=StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    ),
    scoring="accuracy",
    n_jobs=-1
)

svm_cv.fit(X_train, y_train)

best_svm = svm_cv.best_estimator_

best_pred = best_svm.predict(X_test)
best_scores = best_svm.decision_function(X_test)

print("Best C:", svm_cv.best_params_["svm__C"])
print("CV Accuracy:", svm_cv.best_score_)
print("Test Accuracy:", accuracy_score(y_test, best_pred))
print("Test ROC AUC:", roc_auc_score(y_test, best_scores))

Best C: 100
CV Accuracy: 0.5081433224755699
Test Accuracy: 0.509229098805646
Test ROC AUC: 0.5117676739841182


## Final Comparison:

In [30]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

models = [
    ("Majority Baseline",
     dummy.predict(X_test),
     dummy.predict_proba(X_test)[:, 1]),

    ("Linear SVM",
     y_pred_linear,
     y_score_linear),

    ("RBF SVM (C=1)",
     y_pred_rbf,
     y_score_rbf),

    ("RBF SVM (CV-selected)",
     best_pred,
     best_scores)
]

svm_results = pd.DataFrame([
    {
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_test, predictions, zero_division=0
        ),
        "ROC AUC": roc_auc_score(y_test, scores)
    }
    for name, predictions, scores in models
])

svm_results.round(4)

,Model,Accuracy,Precision,Recall,F1,ROC AUC
0,Majority Baseline,0.5014,0.0000,0.0000,0.0000,0.5000
1,Linear SVM,0.5008,0.4995,0.5253,0.5121,0.5067
2,RBF SVM (C=1),0.5100,0.5145,0.3092,0.3863,0.5169
3,RBF SVM (CV-selected),0.5092,0.5093,0.4311,0.4670,0.5118


## Week 5 Conclusions

This week, Support Vector Machines (SVMs) were used to predict whether NBA games would finish Over or Under the sportsbook total. All models used standardized pregame sportsbook features.

The linear SVM achieved 50.08% accuracy and an F1 score of 0.5121. An RBF kernel was then tested to determine whether nonlinear relationships could improve classification. The RBF SVM with C=1 achieved slightly higher accuracy of 51.00%, but its F1 score decreased to 0.3863.

To examine regularization, three-fold cross-validation was used to select the best C value using only the training data. The cross-validation-selected RBF model achieved 50.92% test accuracy and a ROC AUC of 0.5118. These results were only slightly better than the majority-class baseline accuracy of 50.14%.

Overall, neither the linear nor nonlinear SVM demonstrated a meaningful predictive advantage. Hyperparameter tuning also failed to substantially improve performance. These findings suggest that changing the decision boundary and adjusting regularization are not sufficient to reliably predict NBA Over/Under outcomes using the current sportsbook variables. Additional features, such as recent team performance and rest days, may help improve future predictions.